In [0]:
bronze_table="databricks_practice.bronze.bronze_table"
silver_table="databricks_practice.silver"

In [0]:
%sql
-- Create a temporary view with all bronze data plus the test insert

-- Insert test row using SELECT UNION ALL
CREATE OR REPLACE TEMP VIEW bronze_table_test AS
SELECT
    product_id,
    product_name,
    model_number,
    style_color,
    brand_name,
    category,
    subcategory,
    color_name,
    sport_tags,
    product_url,
    canonical_url,
    image_url,
    dt_ingestion_partition,
    dt_ingestion
FROM databricks_practice.bronze.bronze_table
UNION ALL
SELECT
    '0001ca0d-3aa0-36a4-a675-ed169a64b4a8',
    'Old Product Name',
    'MN-001',
    'SC-Red',
    'BrandX',
    'Shoes',
    'Running',
    'Red',
    'Sport1,Sport2',
    'http://example.com/product',
    'http://example.com/canonical',
    'http://example.com/image.jpg',
    DATE('2026-03-30'),
    TIMESTAMP('2026-03-30 08:00:00')

In [0]:
%sql
-- # Test New Record
CREATE OR REPLACE TEMP VIEW bronze_table_test AS
SELECT
    '0001ca0d-3aa0-36a4-a675-ed169a64b4a8' as product_id,
    'New Product Name' as product_name,
    'MN-001' as model_number,
    'SC-Blue' as style_color,
    'BrandX' as brand_name,
    'Shoes' as category,
    'Running' as subcategory,
    'Red' as color_name,
    'Sport3' as sport_tags,
    'http://example.com/product' as product_url,
    'http://example.com/canonical' as canonical_url,
    'http://example.com/image.jpg' as image_url,
    DATE('2026-05-01') as dt_ingestion_partition,
    TIMESTAMP('2026-05-01 08:00:00') as dt_ingestion
FROM databricks_practice.bronze.bronze_table

In [0]:
%sql
-- # Test Old Record
CREATE OR REPLACE TEMP VIEW bronze_table_test AS
SELECT
    '0001ca0d-3aa0-36a4-a675-ed169a64b4a8' as product_id,
    'New Product Name' as product_name,
    'MN-001' as model_number,
    'SC-Blue' as style_color,
    'BrandX' as brand_name,
    'Shoes' as category,
    'Running' as subcategory,
    'Red' as color_name,
    'Sport3' as sport_tags,
    'http://example.com/product' as product_url,
    'http://example.com/canonical' as canonical_url,
    'http://example.com/image.jpg' as image_url,
    DATE('2026-01-01') as dt_ingestion_partition,
    TIMESTAMP('2026-01-01 08:00:00') as dt_ingestion
FROM databricks_practice.bronze.bronze_table

In [0]:
# # Step 1. Order Table
df = spark.sql("""
SELECT
  product_id,
  product_name,
  model_number,
  style_color,
  brand_name,
  category,
  subcategory,
  color_name,
  sport_tags,
  product_url,
  canonical_url,
  image_url,
  dt_ingestion_partition,
  dt_ingestion
FROM bronze_table_test
ORDER BY dt_ingestion_partition DESC
""")

df_dedup = df.dropDuplicates(["product_id"])
df_dedup.createOrReplaceTempView("vw_produto")
vw_produto = df_dedup

In [0]:
display(vw_produto)

In [0]:
# Step 2. Create TABLE Schema
spark.sql(f"""
CREATE OR REPLACE TABLE {silver_table}.dim_product_scd2
(
    product_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    product_id STRING,
    product_name STRING,
    model_number STRING,
    style_color STRING,
    brand_name STRING,
    category STRING,
    subcategory STRING,
    color_name STRING,
    sport_tags STRING,
    product_url STRING,
    canonical_url STRING,
    image_url STRING,
    dt_ingestion_partition DATE,
    dt_ingestion TIMESTAMP,
    start_dt TIMESTAMP,
    end_dt TIMESTAMP,
    current_flag CHAR(1),
    updated_ts TIMESTAMP
)
USING DELTA
""")

In [0]:
# Definir BK, mapear colunas que vao sofrer alteracao (pode ser 2 colunas por exemplo)

In [0]:
# Adicionar insert

In [0]:
# Step 3. Insert data and apply SCD2 history tracking
# Check if the table is empty
if spark.table('databricks_practice.silver.dim_product_scd2').count() == 0:
    
    print(f"Performing initial load into {silver_table}.dim_product_scd2 with SCD2 tracking")
    
    spark.sql(f"""
        INSERT INTO {silver_table}.dim_product_scd2(
            {','.join(vw_produto.columns)},
            start_dt,
            end_dt,
            current_flag,
            updated_ts
        )
        SELECT
            {','.join(vw_produto.columns)},
            current_timestamp() as start_dt,
            NULL as end_dt,
            'Y' as current_flag,
            current_timestamp() as updated_ts
        FROM vw_produto
    """)

else:
    
    print(f"Applying SCD2 logic (MERGE + INSERT) in {silver_table}.dim_product_scd2")

    # 1. CLOSE OLD RECORDS (SCD2 - UPDATE via MERGE)
    spark.sql(f"""
        MERGE INTO {silver_table}.dim_product_scd2 AS target
        USING vw_produto AS source
        ON target.product_id = source.product_id  -- BK
        AND target.current_flag = 'Y' AND target.dt_ingestion < source.dt_ingestion -- filter to only update records with newer dt_ingestion 

        WHEN MATCHED AND (
            target.style_color <> source.style_color OR  -- scd2
            target.category <> source.category OR        -- scd2
            target.color_name <> source.color_name OR    -- scd2
            target.sport_tags <> source.sport_tags       --scd2
        )
        THEN UPDATE SET
            target.end_dt = current_timestamp(),
            target.current_flag = 'N',
            target.updated_ts = current_timestamp()
    """)

    # 2. INSERT NEW VERSIONS AND NEW RECORDS
    spark.sql(f"""
        INSERT INTO {silver_table}.dim_product_scd2 (
            {','.join(vw_produto.columns)},
            start_dt,
            end_dt,
            current_flag,
            updated_ts
        )
        SELECT
            source.product_id,
            source.product_name,
            source.model_number,
            source.style_color,
            source.brand_name,
            source.category,
            source.subcategory,
            source.color_name,
            source.sport_tags,
            source.product_url,
            source.canonical_url,
            source.image_url,
            source.dt_ingestion_partition,
            source.dt_ingestion,
            -- start_dt
            current_timestamp(),

            -- end_dt, if the source is older then close the line with the current dt_ingestion, else NULL (open line)
            CASE 
                WHEN target.product_id IS NOT NULL 
                    AND source.dt_ingestion < target.dt_ingestion 
                THEN target.dt_ingestion
                ELSE NULL
            END,

            -- current_flag, if the dt_ingestion is newer then open the line, else close the line
            CASE 
                WHEN target.product_id IS NULL THEN 'Y'
                WHEN source.dt_ingestion > target.dt_ingestion THEN 'Y'
                ELSE 'N'
            END,
            current_timestamp()

        FROM vw_produto source

        -- LEFT JOIN to get the last version of the record
        LEFT JOIN {silver_table}.dim_product_scd2 target
            ON target.product_id = source.product_id
            AND target.current_flag = 'Y'
        WHERE 
    (
        target.product_id IS NULL -- if the product does not exists
        -- or if the product has any modification in scd2 columns, insert the new line
        OR (
            target.style_color <> source.style_color OR
            target.category <> source.category OR
            target.color_name <> source.color_name OR
            target.sport_tags <> source.sport_tags
        )
        OR source.dt_ingestion < target.dt_ingestion
    )

    -- idempotency - insert records only if not exists
    AND NOT EXISTS (
        SELECT 1
        FROM databricks_practice.silver.dim_product_scd2 t2
        WHERE t2.product_id = source.product_id
          AND t2.dt_ingestion = source.dt_ingestion
          AND t2.style_color = source.style_color
          AND t2.category = source.category
          AND t2.color_name = source.color_name
          AND t2.sport_tags = source.sport_tags
    );
    """)

In [0]:
# Test 1 - First execution
display(spark.table("databricks_practice.silver.dim_product_scd2").count())

In [0]:
# Test 2 - ADD line scd2, new
display(spark.table("databricks_practice.silver.dim_product_scd2").count())

In [0]:
# Test 3 - ADD line scd2, old
display(spark.table("databricks_practice.silver.dim_product_scd2").count())

In [0]:
%sql
-- # Test 1
SELECT * FROM databricks_practice.silver.dim_product_scd2 WHERE product_id = "0001ca0d-3aa0-36a4-a675-ed169a64b4a8" LIMIT 20

In [0]:
%sql
-- Test 2 - New record
SELECT * FROM databricks_practice.silver.dim_product_scd2 WHERE product_id = "0001ca0d-3aa0-36a4-a675-ed169a64b4a8" LIMIT 20

In [0]:
%sql
-- Test 3 - Old Record
SELECT * FROM databricks_practice.silver.dim_product_scd2 WHERE product_id = "0001ca0d-3aa0-36a4-a675-ed169a64b4a8" LIMIT 20

In [0]:
%sql
drop table databricks_practice.silver.dim_product_scd2

In [0]:
# CTE  num_occurences + row_num

In [0]:
# Fact - Update according with dim scd2 and other dims